In [ ]:
import pandas as pd
import json
import numpy as np 
# Load the CSV file into a DataFrame
url = 'DE_youtube_trending_data.csv'
df = pd.read_csv(url)



# --- STEP 1: RENAME ORIGINAL KAGGLE COLUMNS ---
# Renaming your raw columns directly to match the first column of the screenshot
df = df.rename(
    columns={
        "channelTitle": "channel",
        "categoryId": "category_id",
        "publishedAt": "published_at",
        "view_count": "views",
        "comment_count": "comments",  # Matching your "check with df.columns" note
    }
)

# --- STEP 2: DATETIME ENGINEERING ---
# Converting raw string to datetime to extract weekday and hour
#df["published_at"] = pd.to_datetime(df["published_at"])
df["published_at"] = pd.to_datetime(df["published_at"])
df["published_at"] = df["published_at"].dt.tz_convert("Europe/Berlin")
df["publish_weekday"] = df["published_at"].dt.day_name()
df["publish_hour"] = df["published_at"].dt.hour

# --- STEP 3: MASK HIDDEN METRICS (FLOAT WITH NaN) ---
# Cast to float so they can natively accept NaN values if hidden
df["likes"] = df["likes"].astype(float)
df["comments"] = df["comments"].astype(float)

# Wherever ratings or comments are disabled, assign NaN
df.loc[df["ratings_disabled"] == True, "likes"] = None
df.loc[df["comments_disabled"] == True, "comments"] = None

# --- STEP 4: CALCULATE engagement RATES ---

# --- STEP 4: CALCULATE RATES AS PERCENTAGES (*) ---
# Multiplying by 100 converts decimal rates (e.g., 0.052) into percentages (e.g., 5.2)
df["like_rate"] = (df["likes"] / df["views"]) * 100
df["comment_rate"] = (df["comments"] / df["views"]) * 100




# --- STEP 5: EXTERNAL CATEGORY MAPPING ---
try:
    with open("DE_category_id.json", "r") as f:
        category_data = json.load(f)
    category_map = {
        int(item["id"]): item["snippet"]["title"] for item in category_data["items"]
    }
    df["category_name"] = df["category_id"].map(category_map)
except FileNotFoundError:
    df["category_name"] = None

# --- STEP 6: ADD MISSING SCHEMAS ---
df["duration_sec"] = None  # Explicitly marked as "X not available" for Kaggle
df["period"] = "2020-2024"  # Explicitly marked as "2020-2024" for Kaggle

# --- STEP 7: SELECT & ORDER EXACTLY AS SHOWN IN THE SCREENSHOT ---
final_columns = [
    "video_id",
    "title",
    "channel",
    "category_id",
    "category_name",
    "published_at",
    "publish_weekday",
    "publish_hour",
    "views",
    "likes",
    "comments",
    "like_rate",
    "comment_rate",
    "duration_sec",
    "period",
]

    
df = df[final_columns]

# Check your final renamed dataset structure
df.info()
df

# --- STEP 6.5: CLEAN UP REMAINING MISSING VALUES ---

# 1. Fill missing category names from the JSON mapping with "Unknown"
df["category_name"] = df["category_name"].fillna("Unknown")

# 2. Fix the 4 videos with 0 views causing division by zero NaN rates
#df.loc[df["views"] == 0, ["like_rate", "comment_rate"]] = 0.0
df.loc[df["views"] == 0, ["like_rate", "comment_rate"]] = 0.0
df = df[df["views"] > 0]
# Sort by views descending, then drop duplicate video IDs keeping the first (highest view count)
df = df.sort_values(by="views", ascending=False)

df = df.drop_duplicates(subset=["video_id"], keep="first")

  # Sort by date ascending, then drop duplicate video IDs keeping the first instance
#df_final = df_final.sort_values(by="published_at", ascending=True)
#df_final = df_final.drop_duplicates(subset=["video_id"], keep="first")


# 3. (Optional) If you want to fill disabled metrics with 0 instead of NaN, uncomment below:
# df['likes'] = df['likes'].fillna(0)
# df['comments'] = df['comments'].fillna(0)
# df['like_rate'] = df['like_rate'].fillna(0)
# df['comment_rate'] = df['comment_rate'].fillna(0)

# Save your DataFrame to a CSV file
df.to_csv('kaggle_clean.csv', index=False)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 268736 entries, 0 to 268735
Data columns (total 15 columns):
 #   Column           Non-Null Count   Dtype                        
---  ------           --------------   -----                        
 0   video_id         268736 non-null  object                       
 1   title            268736 non-null  object                       
 2   channel          268736 non-null  object                       
 3   category_id      268736 non-null  int64                        
 4   category_name    268590 non-null  object                       
 5   published_at     268736 non-null  datetime64[ns, Europe/Berlin]
 6   publish_weekday  268736 non-null  object                       
 7   publish_hour     268736 non-null  int32                        
 8   views            268736 non-null  int64                        
 9   likes            267489 non-null  float64                      
 10  comments         264550 non-null  float64               

In [31]:
df.duplicated().sum()

np.int64(0)

Breakdown of Your Missing Values
• duration_sec (268,736 missing): Perfect. This matches your total row count exactly, confirming it was successfully initialized 
as completely empty per your schema requirements.
• likes (1,247) & comments (4,186) missing: Perfect. These represent the videos where creators disabled ratings or comments.
• like_rate (1,247) & comment_rate (4,190) missing: Perfect. The 1,247 missing like 
rates match the disabled likes. The comment rate has 4 extra missing values (4,190 vs 4,186)—this means 
you have exactly 4 videos with 0 views, which caused a division-by-zero (NaN).
• category_name (146 missing): This means there are 146 rows with a category_id that does not exist in your DE_category_id.json file.


In [49]:
df['views'].isnull().sum()

np.int64(0)

In [27]:
df['category_name'].unique()

array(['Music', 'Entertainment', 'Sports', 'Howto & Style',
       'Travel & Events', 'Gaming', 'Comedy', 'People & Blogs',
       'Education', 'Autos & Vehicles', 'Science & Technology',
       'Film & Animation', 'News & Politics', 'Pets & Animals', 'Unknown'],
      dtype=object)

In [50]:
df_sorted = df.sort_values(by="views", ascending=True)

In [51]:
df['publish_weekday'].value_counts()

publish_weekday
Friday       9495
Sunday       8562
Wednesday    8184
Tuesday      8024
Thursday     7978
Monday       7342
Saturday     5882
Name: count, dtype: int64

In [53]:
df_sorted

,video_id,title,channel,category_id,category_name,published_at,publish_weekday,publish_hour,views,likes,comments,like_rate,comment_rate,duration_sec,period
32337,xVeLTIhnjPE,PA Sports über Capital Bra und die Zusammenarb...,PA Sports Twitch Highlights,20,Gaming,2021-01-22 00:07:57+01:00,Friday,0,7841,603.0,56.0,7.690346,0.714195,None,2020-2024
60932,1E61MSy041g,FORTNITE SEASON 7! BATTLE PASS & Story Trailer...,iOllek,20,Gaming,2021-06-08 08:37:10+02:00,Tuesday,8,12203,809.0,89.0,6.629517,0.729329,None,2020-2024
21361,07dwGy6xCLY,DIY: Stylische Weihnachtsdeko mit Holz-Cubes s...,Deko Kitchen – Schöne Deko selber machen | Est...,26,Howto & Style,2020-11-25 09:00:00+01:00,Wednesday,9,16185,846.0,62.0,5.227062,0.383071,None,2020-2024
255699,46SOFwr-rR8,Reich werden mit Tik Tok Streams?! Das steckt ...,DER KIOSK,22,People & Blogs,2024-02-05 18:02:00+01:00,Monday,18,16917,857.0,40.0,5.065910,0.236449,None,2020-2024
54318,8W9aFvqU2ks,DAS HASSEN WIR AN UNS! & bei Geld hört die Fre...,Erika,26,Howto & Style,2021-05-11 17:00:09+02:00,Tuesday,17,18700,825.0,59.0,4.411765,0.315508,None,2020-2024
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115917,VK1pC2J57YY,Javelin Balloon Pop Racing Is INTENSE!!,How Ridiculous,17,Sports,2022-03-02 22:00:27+01:00,Wednesday,22,171221005,7621700.0,6066.0,4.451381,0.003543,None,2020-2024
68315,qF0N19MgI3Q,JETSKI WAX PRANK!! 🤣 - #Shorts,Julius Dein,24,Entertainment,2021-07-04 21:00:09+02:00,Sunday,21,176598344,4571088.0,24182.0,2.588409,0.013693,None,2020-2024
149927,gQlMMD8auMs,BLACKPINK - ‘Pink Venom’ M/V,BLACKPINK,10,Music,2022-08-19 06:00:13+02:00,Friday,6,206344829,11945470.0,3291377.0,5.789081,1.595086,None,2020-2024
3152,gdZLi9oWNZg,BTS (방탄소년단) 'Dynamite' Official MV,Big Hit Labels,10,Music,2020-08-21 05:58:10+02:00,Friday,5,219110491,15460845.0,5810618.0,7.056187,2.651912,None,2020-2024
